<a href="https://colab.research.google.com/github/ChaoYuYang0/FibrilGen-v1/blob/main/my_first_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A brief introduction
In this tutorial, we will investigate how FibrilGen can build different peptide alignments and assemble fibril geometries for a given peptide molecule. Specifically, we can control seven parameters: $N$, **K**, $M$, $\theta_s$, $\theta_y$, $r_y$, $\theta_z$ , as illustrated in [Figure 1](https://github.com/ChaoYuYang0/FibrilGen-v1/blob/main/demo/demo-fig1.png).


The goal is to help experimentalists build and visualise possible cross-beta structures and to help computational scientists screen and validate hypothetical cross-beta structures.

<!-- Jas: The science behind the structures, motivation of building them. The goal of the workshop is to let the users understand the parameters and how to use these parameters to build the structures they want. We can run the notebook (code, users' experience) along with the slides (Figures, the guilding materials). -->



# Setup a Python environment
First of all, we setup an environment with essential Python libraries.

<!-- Jas: add package version, and whether or not use pypi -->

In [ ]:
!pip install pymol-open-source==3.2.0a0
!pip install openbabel==3.2.1
!pip install py3Dmol==2.5.5
!git clone https://github.com/ChaoYuYang0/FibrilGen-v1

# Change the working  directory
Once we finish the setup, we change the working directory to "FibrilGen-v1".

In [ ]:
%cd FibrilGen-v1/

# Step 1: build your beta-strand
In this section, we change the input peptide sequence and visualise the output peptide molecule. We change the side-chain rotation angles to see how they can introduce steric clashes (optional).

In [ ]:
Input_seq = 'FEFKFEFK' #@param {type:"string"}
Angle_sidechain = '1_n_ca_cb_cg_-20,3_n_ca_cb_cg_-20,5_n_ca_cb_cg_-20,7_n_ca_cb_cg_-20' #@param {type:"string"}
Ouput_filename = 'F8.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
root = 'demo/Google_Colab/'

## Import libraries
from seq2pep import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Create a b-strand structure
# Initialize a b-strand
get_b_strand(Input_seq)
# Align bulky residues on the cross-section
if Angle_sidechain != "":
  lo_string = Angle_sidechain.split(',')
  for string in lo_string:
    resi,atom1,atom2,atom3,atom4,angle = string.split('_')
    set_chi_angle(resi,atom1,atom2,atom3,atom4,float(angle))
# Save structure
pymol.cmd.save(root+Ouput_filename,'pep')

## Show the structure
if Show_the_structure:
  import py3Dmol
  with open(root+Ouput_filename) as ifile:
      system = "".join([x for x in ifile])
  view = py3Dmol.view(width=900, height=300)
  view.addModelsAsFrames(system)
  view.setStyle({"stick": {'color': 'spectrum'}})
  view.zoomTo()
  view.show()

#Step 2: Build your bilayer structure
In this section, we change the input peptide alignment and visualise the output 2 x 2 beta-sheet building block. We multiply the 2 x 2 building block 5 times, then optimise the assembled structure. We check whether inter-peptide backbone hydrogen bonds and side-chain packing are plausible in the assembled structure.

In [ ]:
Input_filename = 'F8.pdb' #@param {type:"string"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
Alignment_pep_orientations = 'aaa' #@param ['aaa','apa','aap','app','paa','ppa','pap','ppp']
Start_res_at_face = 0 #@param {type:"integer"}
Alignment_face_to_face = 1 #@param {type:"integer"}
Energy_minimisation = 1 #@param {type:"integer"}
Output_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
root = 'demo/Google_Colab/'

from pep2unit import *
from reordering import *
import os

def mol2pdb_convertsion(root,filename):
	os.system(f'obabel {root}{filename}.mol -O {root}{filename}_temp.pdb')
	pymol.cmd.load(f'{root}{filename}_temp.pdb')
	pymol.cmd.save(f'{root}{filename}.pdb',f'{filename}_temp')
	os.system(f'rm {root}{filename}_temp.pdb')

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
name = Input_filename[:-4]
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide, start residue index, end residue index)
unit = create_pep_unit(name,int(resi_start),int(resi_end),Start_res_at_face)

## --- Examples of sheet structures ---
# INPUT: (peptide unit)
sheet = create_sheet(unit)
# INPUT: (peptide alignment aaa/apa/aap/app/paa/ppa/pap/ppp, does two beta-sheet aligned face-to-face, num of units per sheet)
sheet.build_a_plain_sheet(Alignment_pep_orientations,Alignment_face_to_face,5)
## --- End ---

## Save the structure
pymol.cmd.save(root+'bilayer.pdb','plain_sheet')
ordering_indices(root,'bilayer.pdb','bilayer_indexed.pdb')

## Import openbabel and optimization library
if Energy_minimisation:
  from optimize import *
  # Remove steric clashes
  pymol.cmd.create('plain_sheet_em','plain_sheet')
  minimize('plain_sheet_em',nsteps0=10)
  # Save the structure
  pymol.cmd.save(root+'bilayer_em.mol','plain_sheet_em')
  mol2pdb_convertsion(root,'bilayer_em')
  # Index the structure (openbabel energy minimisation changes peptide indices)
  ordering_indices(root,'bilayer_em.pdb','bilayer_em_indexed.pdb')
  reordering_indices(root,'bilayer_indexed','bilayer_em_indexed','bilayer_em_indexed2',20,Num_residues_per_peptide)
  # Calculate the energy
  os.system(f'obabel {root}bilayer_em_indexed2.pdb -otxt --energy --append "Energy" --ff MMFF94')
  # Ouput the structure for visualisation
  os.system(f'cp {root}bilayer_em_indexed2.pdb {root}{Output_filename}')
else:
  # Ouput the structure for visualisation
  os.system(f'cp {root}bilayer.pdb {root}{Output_filename}')

## Show the structure
if Show_the_structure:
  import py3Dmol
  with open(root+Output_filename) as ifile:
      system = "".join([x for x in ifile])
  view = py3Dmol.view(width=900, height=300)
  view.addModelsAsFrames(system)
  view.setStyle({"stick": {'color': 'spectrum'}})
  view.zoomTo()
  view.show()


# Step 3: Build your fibril structure
In this section, we use a 2 x 2 building block to assemble different classes of cross-beta structures.

(a) We use our optimised bilayer to assemble a **sheet** structure. We change the number of stacking (parameter $N$) along the beta-sheet axis and visualise the structure.


In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 10 #@param {type:"integer"}
Output_filename = 'F8_sheet.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
resi_start, resi_end = Reference_segment.split('-')
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a plain sheet (num of units per sheet)
fibril.build_a_flat_sheet(N)

## Save the structure
pymol.cmd.save(root+Output_filename,'plain_sheet')

## Show the structure
if Show_the_structure:
  import py3Dmol
  with open(root+Output_filename) as ifile:
      system = "".join([x for x in ifile])
  view = py3Dmol.view(width=900, height=300)
  view.addModelsAsFrames(system)
  view.setStyle({"stick": {'color': 'spectrum'}})
  view.zoomTo()
  view.show()



(b) We use our optimised bilayer to assemble a **stacked sheet** structure. We change the number of stacking on the sheet cross-section (parameter **K**, do not specify element 1 surrounded by element 0) and the number of stacking (parameter $N$) along the beta-sheet axis.

In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 10 #@param {type:"integer"}
K = '0_1_0,1_1_1' #@param {type:"string"}
Output_filename = 'F8_stacked_sheet.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
#@markdown - Parameter **K** is a binary matrix ("_" to seperate columns, "," to sperate rows) assigning whether a 2 x 2 unit is presented.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a stacked sheet (stacking pattern, num of units per sheet)
# Get parameters
lo_row = K.split(',')
matrix_K = []
for row in lo_row:
  cols = row.split('_')
  matrix_K += [[int(col) for col in cols]]
# Build the structure
fibril.build_a_stacked_sheet(matrix_K,N)

## Save the structure
pymol.cmd.save(root+Output_filename,'plain_sheet')

## Show the structure
if Show_the_structure:
  import py3Dmol
  with open(root+Output_filename) as ifile:
      system = "".join([x for x in ifile])
  view = py3Dmol.view(width=900, height=300)
  view.addModelsAsFrames(system)
  view.setStyle({"stick": {'color': 'spectrum'}})
  view.zoomTo()
  view.show()



(c) We use our optimised bilayer to assemble a **rod** structure. We change the twist chirality (the sign of parameter $\theta_y$), the initial tilt angle (parameter $\theta_z$), and the number of stacking (parameter $N$) along the fibril axis.

In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 50 #@param {type:"integer"}
theta_z = 5.0 #@param {type:"number"}
Sign_theta_y = 1 #@param {type:"integer"}
Output_filename = 'F8_rod.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
#@markdown - Parameter $\theta_z$ is the initial tilt angle.
#@markdown - Parameter "sign of $\theta_y$"" assigned to 1 for left-handed twist, and -1 for right handed twist.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a rod (tilt angle, num of units per sheet, twist sign)
fibril.build_a_rod(theta_z,N,Sign_theta_y)

if fibril.write_output:
  ## Print the refined fibril dimension
  print ('\n')
  print ('##### Refined fibril structure #####')
  fibril.get_dimension()
  print ('#################################### \n')

  ## Save the structure
  pymol.cmd.save(root+Output_filename,'a_rod')

  ## Show the structure
  if Show_the_structure:
    import py3Dmol
    with open(root+Output_filename) as ifile:
        system = "".join([x for x in ifile])
    view = py3Dmol.view(width=900, height=300)
    view.addModelsAsFrames(system)
    view.setStyle({"stick": {'color': 'spectrum'}})
    view.zoomTo()
    view.show()



(d) We use our optimised bilayer to assemble a **stacked rod** structure. We change the number of stacking on the fibril cross-section (parameter **K**, do not specify element 1 surrounded by element 0); the twist chirality (the sign of parameter $\theta_y$), the initial tilt angle (parameter $\theta_z$), and the number of stacking (parameter $N$) along the fibril axis.

In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 10 #@param {type:"integer"}
K = '0_1,1_1' #@param {type:"string"}
theta_z = 10.0 #@param {type:"number"}
Sign_theta_y = 1 #@param {type:"integer"}
Output_filename = 'F8_stacked_rod.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
#@markdown - Parameter **K** is a binary matrix ("_" to seperate columns, "," to sperate rows) assigning whether a 2 x 2 unit is presented.
#@markdown - Parameter $\theta_z$ is the initial tilt angle.
#@markdown - Parameter "sign of $\theta_y$"" assigned to 1 for left-handed twist, and -1 for right handed twist.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a stacked rod (tilt angle, stacking pattern, num of units per sheet, twist sign)
# Get parameters
lo_row = K.split(',')
matrix_K = []
for row in lo_row:
  cols = row.split('_')
  matrix_K += [[int(col) for col in cols]]
# Build the structure
fibril.build_a_stacked_rod(theta_z,matrix_K,N,Sign_theta_y)

if fibril.write_output:
  ## Print the refined fibril dimension
  print ('\n')
  print ('##### Refined fibril structure #####')
  fibril.get_dimension()
  print ('#################################### \n')

  ## Save the structure
  pymol.cmd.save(root+Output_filename,'s_rod')

  ## Show the structure
  if Show_the_structure:
    import py3Dmol
    with open(root+Output_filename) as ifile:
        system = "".join([x for x in ifile])
    view = py3Dmol.view(width=900, height=300)
    view.addModelsAsFrames(system)
    view.setStyle({"stick": {'color': 'spectrum'}})
    view.zoomTo()
    view.show()



(e) We use our optimised bilayer to assemble a **ribbon** structure. We change the twist chirality (the sign of parameter $\theta_y$), the initial tilt angle (parameter $\theta_z$), the initial radius of rotation (parameter $r_y$), and the number of stacking (parameter $N$) along the fibril axis.

In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 50 #@param {type:"integer"}
theta_z = 30.0 #@param {type:"number"}
r_y = 60.0 #@param {type:"number"}
Sign_theta_y = -1 #@param {type:"integer"}
Output_filename = 'F8_ribbon.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
#@markdown - Parameter $\theta_z$ is the initial tilt angle.
#@markdown - Parameter $r_y$ is the initial radius of rotation.
#@markdown - Parameter "sign of $\theta_y$"" assigned to 1 for left-handed twist, and -1 for right handed twist.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a ribbon (tilt angle, radius, num of units per sheet, twist sign)
fibril.build_a_ribbon(theta_z,r_y,N,Sign_theta_y)

if fibril.write_output:
  ## Print the refined fibril dimension
  print ('\n')
  print ('##### Refined fibril structure #####')
  fibril.get_dimension()
  print ('#################################### \n')

  ## Save the structure
  pymol.cmd.save(root+Output_filename,'a_ribbon')

  ## Show the structure
  if Show_the_structure:
    import py3Dmol
    with open(root+Output_filename) as ifile:
        system = "".join([x for x in ifile])
    view = py3Dmol.view(width=900, height=300)
    view.addModelsAsFrames(system)
    view.setStyle({"stick": {'color': 'spectrum'}})
    view.zoomTo()
    view.show()



(f) We use our optimised bilayer to assemble a **stacked ribbon** structure. We change the number of stacking (parameter $M$) and the angle for stacking (parameter $\theta_s$) on the fibril cross-section; the twist chirality (the sign of parameter $\theta_y$), the initial tilt angle (parameter $\theta_z$), the initial radius of rotation (parameter $r_y$), and the number of stacking (parameter $N$) along the fibril axis.

In [ ]:
Input_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Input_num_peptides_per_sheet = 10 #@param {type:"integer"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
N = 20 #@param {type:"integer"}
theta_s = 100.0 #@param {type:"number"}
M = 2 #@param {type:"integer"}
theta_z = 10.0 #@param {type:"number"}
r_y = 60.0 #@param {type:"number"}
Sign_theta_y = 1 #@param {type:"integer"}
Output_filename = 'F8_stacked_ribbon.pdb' #@param {type:"string"}
Show_the_structure = 1 #@param {type:"integer"}
#@markdown - Parameter $N$ is an integer repeating stacking along the vertical axis.
#@markdown - Parameter $\theta_s$ is angular displacement of units on the fibril cross-section.
#@markdown - Parameter $M$ is the number of stacking on the fibril cross-section.
#@markdown - Parameter $\theta_z$ is the initial tilt angle.
#@markdown - Parameter $r_y$ is the initial radius of rotation.
#@markdown - Parameter "sign of $\theta_y$"" assigned to 1 for left-handed twist, and -1 for right handed twist.
root = 'demo/Google_Colab/'

from builder import *

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(root+Input_filename)

## Create a periodic unit
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide length, the number of peptides per sheet, start residue index, end residue index)
unit = create_sheet_unit(Num_residues_per_peptide,Input_num_peptides_per_sheet,int(resi_start),int(resi_end))

## Create a fibril object
fibril = create_fibril(unit)

## Build a stacked ribbon (tilt angle, radius, stacking angle, stacking number, num of units per sheet, twist sign)
fibril.build_a_stacked_ribbon(theta_z,r_y,theta_s,M,N,Sign_theta_y)

if fibril.write_output:
  # Print the refined fibril dimension
  print ('\n')
  print ('##### Refined fibril structure #####')
  fibril.get_dimension()
  print ('#################################### \n')

  ## Save the structure
  pymol.cmd.save(root+Output_filename,'s_ribbon')

  ## Show the structure
  if Show_the_structure:
    import py3Dmol
    with open(root+Output_filename) as ifile:
        system = "".join([x for x in ifile])
    view = py3Dmol.view(width=900, height=300)
    view.addModelsAsFrames(system)
    view.setStyle({"stick": {'color': 'spectrum'}})
    view.zoomTo()
    view.show()

